In [1]:
from GARCH import get_garch_output
from GARCH import diagnose_z_values
from GARCH import garch_1_1_volatility_forecast
from GARCH import deterministic_portfolio_return_forecasts
import numpy as np

In [2]:
#past returns training data
#default values
interval = "1d"
period = "5y"
price = "Close"

#optional GARCH starting para
#default for omega 0.05 * var(returns)
starting_para = {
    "alpha": 0.5,
    "beta": 0.9,
    "omega": 0.05
}

assets = ["NVO", "AAPL", "SAP", "SI", "NVDA"]

output = get_garch_output(tickers=assets, interval=interval, period=period, price=price, starting_para=starting_para)

In [3]:
output["NVO"]

{'alpha': np.float64(0.09865256261983753),
 'beta': np.float64(0.7728950409049498),
 'omega': np.float64(0.7281595508911031),
 'nu': np.float64(3.7607665411120683),
 'implied unconditional variance': np.float64(5.66871129595169),
 'sample unconditional variance': np.float64(6.071357439728401),
 'expected return': np.float64(0.021900743222022675),
 'return last period': np.float64(2.7285266041963823),
 'conditional vola t+1': np.float64(4.978764680491329),
 'past_z_values': Date
 2021-07-07 00:00:00-04:00    0.614874
 2021-07-08 00:00:00-04:00    0.429063
 2021-07-09 00:00:00-04:00   -0.311523
 2021-07-12 00:00:00-04:00    0.022818
 2021-07-13 00:00:00-04:00    0.023504
                                ...   
 2026-06-26 00:00:00-04:00    0.361689
 2026-06-29 00:00:00-04:00    0.232808
 2026-06-30 00:00:00-04:00   -0.385757
 2026-07-01 00:00:00-04:00    0.792066
 2026-07-02 00:00:00-04:00    1.279248
 Name: Close, Length: 1253, dtype: float64}

In [4]:
nvo_z_values = output["NVO"]["past_z_values"]
nvo_nu = output["NVO"]["nu"]
diagnose_z_values(nvo_z_values, nvo_nu)

mean z values: 0.0028
std z values: 1.0486
estimated degrees of freedom smaller or equal to 4, infinite kurtosis
Ljung-Box-Test for z values
      lb_stat  lb_pvalue
10   5.990527   0.816058
20  14.686319   0.794066
Ljung-Box-Test for z^2 values
     lb_stat  lb_pvalue
10  1.355712   0.999319
20  8.713515   0.985994


In [5]:
aapl_z_values = output["AAPL"]["past_z_values"]
aapl_nu = output["AAPL"]["nu"]
diagnose_z_values(aapl_z_values, aapl_nu)

mean z values: -0.0027
std z values: 0.9929
empirical excess kurtosis z values: 2.839551310965894
theoretical excess kurtosis z values: 8.473981319422721
Ljung-Box-Test for z values
      lb_stat  lb_pvalue
10   7.911877   0.637444
20  11.029405   0.945456
Ljung-Box-Test for z^2 values
      lb_stat  lb_pvalue
10  13.496739   0.197209
20  18.531513   0.552438


In [8]:
#volatility forecast
# default forecast period
forecast_period = 5
garch_1_1_volatility_forecast(output, forecast_period)

In [12]:
#compare to arch package
from arch import arch_model
from GARCH import compute_log_returns
import yfinance as yf

nvo_returns = yf.Ticker("NVO").history(period=period, interval=interval)[price]
log_returns = compute_log_returns(nvo_returns)

model = arch_model(log_returns[0], p=1, q=1).fit()
print(model.summary())


Iteration:      1,   Func. Count:      6,   Neg. LLF: 7405.019066780364
Iteration:      2,   Func. Count:     15,   Neg. LLF: 5073.071972473679
Iteration:      3,   Func. Count:     23,   Neg. LLF: 2994.6865680786677
Iteration:      4,   Func. Count:     30,   Neg. LLF: 2881.019214808426
Iteration:      5,   Func. Count:     36,   Neg. LLF: 2868.000859771052
Iteration:      6,   Func. Count:     42,   Neg. LLF: 2868.5611890646232
Iteration:      7,   Func. Count:     48,   Neg. LLF: 2867.818692554921
Iteration:      8,   Func. Count:     53,   Neg. LLF: 2867.818691265913
Iteration:      9,   Func. Count:     57,   Neg. LLF: 2867.818691266003
Optimization terminated successfully    (Exit mode 0)
            Current function value: 2867.818691265913
            Iterations: 9
            Function evaluations: 57
            Gradient evaluations: 9
                     Constant Mean - GARCH Model Results                      
Dep. Variable:                  Close   R-squared:              

In [7]:
import numpy as np
np.sqrt(model.forecast(horizon=forecast_period).variance.values[0])

array([2.227006  , 2.27044457, 2.30972974, 2.3453301 , 2.37764721])

In [9]:
#forecast scenario based returns
#define scenarios for example to show path dependencies on total returns
s1 = [0.3, -0.2, -0.1, 0.2]
s2 = [-0.2, -0.1, 0.2, 0.3]
s3 = [0.3, 0.2, -0.1, -0.2]
scenarios = [s1, s2, s3]

#define asset (main asset) for which these scenarios are assumed
main_asset = "NVO"

#define portfolio weights for each asset
weights = {
    "NVO": 0.3,
    "AAPL": 0.2,
    "SAP": 0.1,
    "SI": 0.2,
    "NVDA": 0.1
    }

deterministic_portfolio_return_forecasts(garch_output = output, scenarios=scenarios, main_asset=main_asset, asset_weights=weights)